In [ ]:
!pip  install langchain langchain-core langchain-community

In [ ]:
!pip install pypdf pymupdf  sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}

)

In [ ]:
sample_doc

In [4]:
Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [4]:
# Load Text Data 
from langchain_community.document_loaders.text import TextLoader

text_loader = TextLoader('data/Python.txt', encoding="utf-8")

document= text_loader.load()

document

C:\Users\Admin\AppData\Local\Temp\ipykernel_23784\1959413273.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


[Document(metadata={'source': 'data/Python.txt'}, page_content=' Introduction to Python\n\nPython is a high-level, interpreted programming language. It is easy to learn and widely used in web development, data science, artificial intelligence, machine learning, and automation.\n\nPython provides simple syntax, a large collection of libraries, and strong community support. It is suitable for both beginners and experienced programmers.\n\nExample:\n\n\nprint("Hello, World!")\n\n\nThis program displays the message "Hello, World!" on the screen.\n')]

In [ ]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document


# # PDF data USING PYMUPDF LOADER
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

### Ingestion Pipeline

In [5]:
# data into documents 
import os 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [6]:
# Converted data into documents 
def load_all_pdfs():
    folder_path = 'data/pdfs'
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):

        if filename.lower().endswith(".pdf"):
            # Create a complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print(f"Total PDFs: {num_docs}")
    print(f"All pages: {len(all_docs)}")

    return all_docs


In [8]:
all_pdf_documents= load_all_pdfs()

Total PDFs: 2
All pages: 12


### covert data into chunks 

In [10]:
# covert data into chunks 
!pip install langchain-text-splitters

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # created chunks

def split_docs(documents,chunk_size=600,chunk_overlap=50):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chuncked_docs=text_splitter.split_documents(documents)
    return chuncked_docs
    
    

In [12]:
chunks=split_docs(all_pdf_documents)

In [13]:
len(chunks)

91

### Chunks into Embeddings  

In [14]:
# Chunks into Embeddings  
from sentence_transformers import SentenceTransformer

In [15]:


# For creating embeddings
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name

        print("Loading model:", self.model_name)

        # Load the embedding model
        self.model = SentenceTransformer(self.model_name)

        print(
            "Embedding dimensions =",
            self.model.get_sentence_embedding_dimension()
        )

    def generate_embeddings(self, text):
        # Generate embeddings without displaying a progress bar
        embeddings = self.model.encode(
            text,
            show_progress_bar=False
        )

        print("Embeddings shape:", embeddings.shape)

        return embeddings


In [ ]:
embedding_manager = EmbeddingManager()

loading model. all-MiniLM-L6-v2


C:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

 ## Vector Store

In [ ]:
import chromadb
import uuid

In [ ]:

import os
import uuid
import chromadb

class VectorStoreManager:
    def __init__(
        self,
        persist_directory="data/vector_store",
        collection_name="pdf_documents"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for PDF embeddings in RAG"
            }
        )

        print("Initialized vector store:", self.collection_name)
        print("Existing documents:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents does not match number of embeddings"
            )

        if not documents:
            print("No documents to add.")
            return

        ids = [
            f"doc_{uuid.uuid4()}"
            for _ in documents
        ]

        all_metadata = []
        documents_content = []

        for i, doc in enumerate(documents):
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)
            documents_content.append(doc.page_content)

        embeddings_list = embeddings.tolist()

        # Add the entire batch once, outside the loop.
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("Documents added:", len(documents_content))
        print("Total documents in collection:", self.collection.count())


In [ ]:
vector_store=VectorStoreManager()

In [ ]:

# Documents -> Chunks -> Embeddings -> Vector Store

texts = [doc.page_content for doc in chunks]

print("Number of chunks:", len(texts))

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Add embeddings to ChromaDB
vector_store.add_documents(chunks, embeddings)


## Retrieval Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:

class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # Convert the query into an embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Check how many documents are stored
        total_docs = self.vector_store.collection.count()

        if total_docs == 0:
            print("The vector store is empty.")
            return []

        # Retrieve the most relevant documents
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(top_k, total_docs)
        )

        retrieved_docs = []

        # Check whether any documents were retrieved
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            # Process retrieved documents
            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):
                # Convert cosine distance to similarity
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print("Retrieved documents:", len(retrieved_docs))

        else:
            print("No valid documents retrieved.")

        return retrieved_docs


In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
rag_retriever.retrieve("Which scale doctors used to measure the x ray for knee")

### LLM Integration 


In [ ]:
# 1- OPEN AI 
API_KEY_OPENAI="MY API KEY"

In [ ]:
!pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    open_api_key=API_KEY_OPENAI,
    model='gpt-6-astra',
    temperature=0.1,
    max_tokens=1024
)


In [ ]:
# generate the  Retrieval augmented output

def generateOutput(query,retriever,llm, top_k=3):
    results= retriever.retrieve(query,top_k)

    # Build the context join the results comming from databse baes on query 
    context="\n".join([doc["document"  for doc in results ]]) if results else ""

    if not context:
        print("we found no relevant context for given query ")

    # context +query 
    prompt= f""" use  given context to generate the answer  for the query 
                 Context: {context} 
                 Query : {query} """
    response= llm.invoke(prompt) # expected string as input 
    return response.content
    



In [ ]:
answer= generateOutput('What is KOA',rag_retriever, llm)

In [ ]:
### Groq

In [ ]:
API_Key_GROQ = "paste-your-api-key-here"
# !pip install langchain-groq

from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)


In [ ]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)

In [ ]:
print(answer)

In [ ]:
# Claude
# !pip install langchain-anthropic